[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_GPU/Distributed_Training_2.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Distributed Training II

Where [Scale_NN's](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) 'GPU toolbox map' becomes runnable: real multi-process training with torch.distributed (the CPU `gloo` backend — same API as multi-GPU NCCL), the all-reduce that powers it, and the sharding arithmetic of ZeRO/FSDP. The data-parallel gradient is verified equal to the single-process big batch.

## 1. Pre-requisites

[Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb), [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) (processes!). Runs on any multi-core machine — the backend swaps to `nccl` unchanged on a GPU cluster.

---
### 🕐 Session 1 of 3 — *Collectives: All-Reduce & Friends* (~35 min)
**Goal:** the communication primitives; ring all-reduce's bandwidth optimality, derived.
**Builds on:** [OS workshop](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb). &nbsp; **Feeds into:** Session 2 (data parallelism, verified).

---

## 2. The Vocabulary of Together

💡 **Intuition.** Distributed training reduces to a handful of **collectives**: broadcast (one → all), all-reduce (everyone ends with the sum), all-gather, reduce-scatter. The star is all-reduce — gradient averaging *is* one all-reduce per step. The naive way (everyone sends to a chief, chief sends back) moves $2(N{-}1)$ copies of the data through one link. **Ring all-reduce** pipelines chunks around a ring: every link busy, total traffic per node $2\frac{N-1}{N}\times$ the data — *independent of N*. That factor is why thousand-GPU training is possible at all.

In [ ]:
%%writefile dist_worker.py
# the worker script each process runs — the pattern for ALL torch.distributed work
import torch, torch.nn as nn, torch.distributed as dist, os, sys

rank, world = int(sys.argv[1]), int(sys.argv[2])
os.environ.setdefault("MASTER_ADDR", "127.0.0.1")
os.environ.setdefault("MASTER_PORT", "29531")
dist.init_process_group("gloo", rank=rank, world_size=world)     # "nccl" on GPUs — only change

# --- demo 1: all_reduce ---
t = torch.ones(4) * (rank + 1)
dist.all_reduce(t)                                               # in-place sum across workers
if rank == 0:
    print(f"[all_reduce] {world} workers → {t.tolist()}  (= sum 1..{world} in every slot)")

# --- demo 2: data-parallel gradient == big-batch gradient ---
torch.manual_seed(0)                                             # SAME init on every worker
model = nn.Linear(8, 1)
full_x = torch.randn(64, 8, generator=torch.Generator().manual_seed(1))
full_y = torch.randn(64, 1, generator=torch.Generator().manual_seed(2))
shard = slice(rank*64//world, (rank+1)*64//world)                # each worker: its shard

loss = ((model(full_x[shard]) - full_y[shard])**2).mean()
loss.backward()
for p in model.parameters():                                     # DDP-by-hand: average gradients
    dist.all_reduce(p.grad)
    p.grad /= world

if rank == 0:
    ref = nn.Linear(8, 1)
    torch.manual_seed(0); ref = nn.Linear(8, 1)                  # identical init
    ((ref(full_x) - full_y)**2).mean().backward()
    gap = max((p.grad - q.grad).abs().max().item()
              for p, q in zip(model.parameters(), ref.parameters()))
    print(f"[DDP oracle] max |sharded-averaged grad − big-batch grad| = {gap:.2e}")
dist.destroy_process_group()

In [ ]:

# YOUR CODE HERE


**What just happened.** Four independent OS processes started, found each other over a socket, and produced two lines:

```
[all_reduce] 4 workers → [10.0, 10.0, 10.0, 10.0]
[DDP oracle] max |sharded-averaged grad − big-batch grad| = 8.94e-08
```

**The first is a functional check, and it reads exactly as designed.** Worker $r$ contributed a tensor of $(r+1)$, and $1+2+3+4 = 10$ — appearing in **every slot on every worker**. Not a gather to rank 0, not a reduction to one place: **everyone ends holding the sum**, which is what the "all" means.

**The second is the one that matters, and it is a mathematical claim verified numerically.** $8.94\times10^{-8}$ sits at float32 machine epsilon ($\varepsilon \approx 1.2\times10^{-7}$). **Sharding a 64-sample batch across four workers and averaging their gradients gives the same gradient as computing the whole batch in one place** — not approximately, but to the limit of the arithmetic.

**That identity is the entire correctness story of data-parallel training, and it is one line of algebra.**

$$\nabla \Big(\tfrac{1}{64}\textstyle\sum_{i=1}^{64}\ell_i\Big) \;=\; \tfrac{1}{4}\sum_{r=0}^{3} \nabla\Big(\tfrac{1}{16}\textstyle\sum_{i \in \text{shard}_r}\ell_i\Big)$$

**Differentiation is linear, so splitting the sum changes nothing.** It is precisely the gradient-accumulation argument from [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) Session 2 — the same theorem with a network between the terms instead of a `for` loop.

**Note that the residual is *not* zero, and that this is the right outcome.** Four partial sums combined in a different order from one long sum produce different rounding, because **floating-point addition is not associative**. $10^{-8}$ on gradients of order 1 is exactly what float32 predicts. **An exact 0.0 would be more suspicious**, since it would suggest the two paths were not genuinely independent.

**Two lines in the worker script are what make the check meaningful; point at them.** `torch.manual_seed(0)` before building the model gives all four workers **identical initial weights** — averaging gradients from models that disagree is meaningless. And the explicit `Generator().manual_seed(...)` means each worker constructs the *same* 64 samples before taking its own slice. **Distributed correctness is mostly about controlling precisely what is shared and what is not.**

**Note also what this hand-rolled version leaves out, because real DDP adds exactly one thing.** Here every gradient is all-reduced *after* the whole backward pass finishes. Production DDP all-reduces **bucket by bucket as backprop produces each gradient**, overlapping communication with the remaining backward computation — **the streams idea from [HW-Accelerated Computing](./HW_Accelerated_Computing.ipynb), at cluster scale.** Same mathematics, better choreography.

**And note the wall that choreography eventually hits.** Per-step communication is roughly **2 bytes per parameter** (fp16 gradients), independent of batch size. Shrink the step time enough — more workers, smaller per-worker batch — and transfer time exceeds compute time, at which point **adding workers stops helping**. That is the [roofline](./Performance_Engineering.ipynb) argument with a network on the memory axis.

**Finally, the practical note for re-running this cell.** `init_process_group` **blocks until all four processes arrive**, so a worker that crashes early makes the others hang rather than fail. A timeout here usually means a stale process still holds port 29531. **A hang, not an exception, is the characteristic failure of collective code** — worth meeting on a laptop rather than on a cluster.

---
### 🕐 Session 2 of 3 — *Data Parallelism, Verified* (~35 min)
**Goal:** what DDP actually does per step; the overlap trick; when communication becomes the wall.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (sharding: ZeRO/FSDP).

---

## 3. What You Just Proved

💡 **Intuition.** The oracle line above is the entire correctness story of DDP: sharding the batch and averaging gradients is *mathematically identical* to one big batch ([gradient accumulation](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb)'s twin, across machines). Real DDP adds one performance trick: gradients are all-reduced **bucket by bucket as backprop produces them**, overlapping communication with the rest of the backward pass — the [streams idea](./HW_Accelerated_Computing.ipynb) at cluster scale. The wall: per-step communication is ~2 bytes/param (fp16 grads); when step time shrinks below transfer time, scaling stalls — the [roofline](./Performance_Engineering.ipynb) with a network axis.

---
### 🕐 Session 3 of 3 — *Sharding: the ZeRO/FSDP Arithmetic* (~30 min)
**Goal:** when the MODEL doesn't fit: shard optimizer states, gradients, weights — the memory ladder.
**Builds on:** Session 2.

---

## 4. The Memory Ladder

[Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) counted Adam-fp32 training at ≈16 bytes/param. Sharding across $N$ workers divides what it can:

| Stage | Shards | bytes/param/worker (N=8) |
|---|---|---|
| DDP (none) | — | 16 |
| ZeRO-1 | optimizer states (8B) | 8 + 8/8 = 9 |
| ZeRO-2 | + gradients (4B) | 4 + 12/8 = 5.5 |
| ZeRO-3 / FSDP | + weights (4B) | 16/8 = 2 |

💡 **Intuition.** Each rung trades memory for *communication choreography*: ZeRO-3 must all-gather each layer's weights just-in-time for its forward/backward, then drop them — streaming the model through workers the way [overlap-save](../Intro_DSP/Foundations_of_Signal_Processing_1.ipynb) streams a signal through a filter. Pipeline and tensor parallelism split along different axes (layers; within-matmul) and compose with all of the above — the '3-D parallelism' of frontier training runs.

**Exercise:** extend `dist_worker.py` to ZeRO-1 — keep Adam moments for only your shard of parameters, all-gather updated weights after each step, and verify training curves match plain DDP.

---
## Where next

- [Scale_NN](../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) — the single-node accounting.
- [Federated Learning](../Intro_Mach_Learn/Federated_Learning_Privacy.ipynb) — averaging for privacy instead of speed.
- [HW-Accelerated Computing](./HW_Accelerated_Computing.ipynb) — the intra-GPU story.